<div style="border-left: 5px solid #4285f4; padding: 16px 20px; margin: 6px 0 20px 0; border-radius: 6px; background: rgba(66,133,244,0.06);">
<h1>✨ Mejorar imágenes con IA</h1>
<p><strong>Restauración de fotografías con CodeFormer</strong></p>
<p>Un flujo sencillo para recuperar detalles, mejorar rostros y ampliar tus imágenes.</p>
<hr>
<h3>🧭 Cómo usar este cuaderno</h3>
<ol>
<li>⚙️ Preparar el entorno.</li>
<li>📤 Subir una foto o seleccionar una que ya tengas guardada.</li>
<li>🎚️ Ajustar la fidelidad, la ampliación y las mejoras opcionales.</li>
<li>🚀 Restaurar la imagen.</li>
<li>🖼️ Comparar el antes y el después.</li>
<li>📥 Descargar el resultado.</li>
</ol>
<p><strong>💡 Recomendación:</strong> selecciona una GPU en <em>Entorno de ejecución → Cambiar tipo de entorno de ejecución</em> para acelerar el procesamiento. Si Colab reinicia el entorno, vuelve a ejecutar la preparación.</p>
</div>

<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>⚙️ 01 · Preparar el entorno</h2>
</div>

Ejecuta esta celda primero. Prepara las dependencias, conecta Google Drive y comprueba que estén disponibles los archivos necesarios para restaurar imágenes.

<blockquote>⏳ La primera ejecución puede tardar unos minutos mientras se preparan los componentes.</blockquote>

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, urllib.request, time
from PIL import Image
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

PROYECTO = Path('/content/Mejorar_Imagenes_con_IA')
ENTRADA = PROYECTO / 'entrada'
RESULTADOS = PROYECTO / 'resultados'
EXTENSIONES = {
    '.png', '.jpg', '.jpeg', '.webp',
    '.bmp', '.tif', '.tiff',
}

# Montar Google Drive si estamos en Colab.
EN_COLAB = False
files = None
try:
    from google.colab import drive, files
    drive.mount('/content/drive', force_remount=False)
    DRIVE_PROYECTO = Path(
        '/content/drive/MyDrive/Mejorar_Imagenes_con_IA'
    )
    DRIVE_ORIGINALES = DRIVE_PROYECTO / 'originales'
    DRIVE_RESULTADOS = DRIVE_PROYECTO / 'resultados'
    DRIVE_ORIGINALES.mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTADOS.mkdir(parents=True, exist_ok=True)
    EN_COLAB = True
except Exception as e:
    print('Aviso de Google Drive/Colab:', repr(e))

for carpeta in (ENTRADA, RESULTADOS):
    carpeta.mkdir(parents=True, exist_ok=True)

# Buscar un repositorio ya preparado antes de descargar uno nuevo.
candidatos_repo = [
    PROYECTO / 'CodeFormer',
    Path('/content/CodeFormer'),
    Path('/content/drive/MyDrive/Mejorar_Imagenes_con_IA')
    / 'CodeFormer',
]
CODEFORMER = next(
    (p for p in candidatos_repo
     if (p / 'inference_codeformer.py').is_file()),
    None,
)

if CODEFORMER is None:
    CODEFORMER = PROYECTO / 'CodeFormer'
    if CODEFORMER.exists():
        shutil.rmtree(CODEFORMER)
    print('Descargando el repositorio oficial de CodeFormer…')
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/sczhou/CodeFormer.git',
         str(CODEFORMER)],
        check=True,
    )
    print('Instalando dependencias. Puede tardar unos minutos…')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-r',
         str(CODEFORMER / 'requirements.txt')],
        check=True,
    )
else:
    print('Repositorio encontrado:', CODEFORMER)

SCRIPT = CODEFORMER / 'inference_codeformer.py'
if not SCRIPT.is_file():
    mensaje = 'No se encontró inference_codeformer.py en '
    raise FileNotFoundError(mensaje + str(CODEFORMER))

# Algunas copias no incluyen version.py.
# BasicSR lo importa al iniciar.
version_py = CODEFORMER / 'basicsr' / 'version.py'
if (CODEFORMER / 'basicsr').is_dir() and not version_py.exists():
    version_py.write_text(
        "__version__ = '0.0.0-local'\n__gitsha__ = 'unknown'\n",
        encoding='utf-8',
    )

# Descargar pesos desde los assets oficiales de GitHub Releases.
# No ejecutar download_pretrained_models.py: importa BasicSR
# antes de descargar
# y puede fallar por incompatibilidades aunque las URL funcionen.
def descargar_peso(url, destino, minimo_bytes=100_000):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.is_file() and destino.stat().st_size >= minimo_bytes:
        print(
            f'Peso ya disponible: {destino.name} ' \
            f'({destino.stat().st_size:,} bytes)'
        )
        return
    temporal = destino.with_suffix(destino.suffix + '.part')
    if temporal.exists():
        temporal.unlink()
    print(f'Descargando {destino.name}…')
    try:
        req = urllib.request.Request(
            url, headers={'User-Agent': 'Mozilla/5.0'}
        )
        with urllib.request.urlopen(req, timeout=90) as respuesta, \
                open(temporal, 'wb') as salida:
            total = int(
                respuesta.headers.get('Content-Length', '0') or 0
            )
            descargado = 0
            while True:
                bloque = respuesta.read(1024 * 1024)
                if not bloque:
                    break
                salida.write(bloque)
                descargado += len(bloque)
                if total:
                    print(f'  {descargado/total:.0%}', end='\r')
        if temporal.stat().st_size < minimo_bytes:
            raise RuntimeError(
                'El archivo descargado es demasiado pequeño '
                f'({temporal.stat().st_size} bytes); ' 
                'puede ser una página de error.'
            )
        temporal.replace(destino)
        print(
            f'Listo: {destino} ' 
            f'({destino.stat().st_size:,} bytes)'
        )
    except Exception as e:
        if temporal.exists():
            temporal.unlink()
        raise RuntimeError(
            f'No se pudo descargar {destino.name} ' 
            f'desde {url}. Detalle: {e}'
        ) from e

BASE_URL = (
    'https://github.com/sczhou/CodeFormer/releases/'
    'download/v0.1.0/'
)
PESO_CODEFORMER = (
    CODEFORMER / 'weights' / 'CodeFormer' / 'codeformer.pth'
)
PESO_DETECCION = (
    CODEFORMER / 'weights' / 'facelib'
    / 'detection_Resnet50_Final.pth'
)
PESO_PARSING = (
    CODEFORMER / 'weights' / 'facelib' / 'parsing_parsenet.pth'
)

descargar_peso(
    BASE_URL + 'codeformer.pth',
    PESO_CODEFORMER,
    minimo_bytes=100_000_000,
)
descargar_peso(
    BASE_URL + 'detection_Resnet50_Final.pth',
    PESO_DETECCION,
    minimo_bytes=1_000_000,
)
descargar_peso(
    BASE_URL + 'parsing_parsenet.pth',
    PESO_PARSING,
    minimo_bytes=1_000_000,
)

print('\nPreparación lista.')
print('Proyecto:', PROYECTO)
print('Repositorio CodeFormer:', CODEFORMER)
print('Entrada:', ENTRADA)
print('Resultados:', RESULTADOS)
try:
    import torch
    print('PyTorch:', torch.__version__)
    gpu_nombre = (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else 'no detectada'
    )
    print(
        'GPU disponible:', torch.cuda.is_available(), '-', gpu_nombre
    )
except Exception as e:
    print('No se pudo comprobar CUDA:', repr(e))

def listar_imagenes():
    return sorted(
        [p for p in ENTRADA.iterdir()
         if p.is_file() and p.suffix.lower() in EXTENSIONES],
        key=lambda p: p.name.lower(),
    )


<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>📤 02 · Elegir una fotografía</h2>
</div>

Sube <strong>una imagen por vez</strong> o selecciona una fotografía que ya esté guardada en la carpeta de entrada.

Formatos habituales compatibles: JPG, JPEG, PNG, WEBP, BMP y TIFF.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

boton_subir = widgets.Button(
    description='Subir imagen',
    icon='upload',
    button_style='primary',
    layout=widgets.Layout(width='190px', height='40px'),
)
selector_imagen = widgets.Dropdown(
    options=[], description='Imagen:',
    layout=widgets.Layout(width='95%'),
)
estado_subida = widgets.Output()

def actualizar_selector(seleccion=None):
    imagenes = listar_imagenes()
    selector_imagen.options = [(p.name, str(p)) for p in imagenes]
    if seleccion and any(str(p) == seleccion for p in imagenes):
        selector_imagen.value = seleccion
    elif imagenes and selector_imagen.value not in [
        str(p) for p in imagenes
    ]:
        selector_imagen.value = str(imagenes[-1])
    elif not imagenes:
        selector_imagen.value = None

def subir_una_imagen(_):
    with estado_subida:
        clear_output()
        if not EN_COLAB:
            print(
                'La subida directa está disponible ' 
                'en Google Colab. '

                'Copia la imagen a:', ENTRADA
            )
            return
        carga = files.upload()
        if not carga:
            print('No se seleccionó ninguna imagen.')
            return
        # Si se eligen varios archivos, solo se guarda el primero.
        nombre, contenido = next(iter(carga.items()))
        ext = Path(nombre).suffix.lower()
        if ext not in EXTENSIONES:
            print(
                'Formato no compatible: '
                f'{ext or "(sin extensión)"}'
            )
            return
        destino = ENTRADA / Path(nombre).name
        destino.write_bytes(contenido)
        if 'DRIVE_ORIGINALES' in globals():
            shutil.copy2(destino, DRIVE_ORIGINALES / destino.name)
        actualizar_selector(str(destino))
        print('Imagen guardada:', destino)

boton_subir.on_click(subir_una_imagen)
actualizar_selector()
display(
    widgets.HBox([boton_subir]), selector_imagen, estado_subida
)


<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>🎚️ 03 · Ajustar la restauración</h2>
</div>

Personaliza el resultado con los controles de la celda siguiente.

- **Fidelidad:** equilibra la conservación de la fotografía original y la reconstrucción de detalles. Un valor menor permite más cambios; uno mayor prioriza conservar la imagen.
- **Ampliación 2× o 4×:** elige el tamaño final. La opción 4× produce imágenes más grandes y puede tardar más.
- **🌄 Mejora del fondo:** activa Real-ESRGAN para mejorar las zonas que no son rostros.
- **🙂 Mejora adicional de rostros:** activa el procesamiento adicional de los rostros detectados.

💡 **Para empezar:** prueba una fidelidad de 0,7 y ampliación 2×. Activa las mejoras adicionales si buscas más detalle; pueden aumentar el tiempo de procesamiento y el uso de memoria de la GPU.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

control_fidelidad = widgets.FloatSlider(
    value=0.7,
    min=0.0,
    max=1.0,
    step=0.05,
    description='Fidelidad:',
    readout_format='.2f',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%'),
)
control_escala = widgets.Dropdown(
    options=[('2×', 2), ('4×', 4)],
    value=2,
    description='Ampliación:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
)
control_fondo = widgets.Checkbox(
    value=False,
    description='Mejorar también el fondo (Real-ESRGAN)',
    indent=False,
    layout=widgets.Layout(width='100%'),
)
control_rostros = widgets.Checkbox(
    value=False,
    description='Aplicar mejora adicional a los rostros',
    indent=False,
    layout=widgets.Layout(width='100%'),
)

panel_mejoras = widgets.VBox(
    [
        widgets.HTML(
            '<b>Opciones que tardan más (opcionales)</b>'
        ),
        control_fondo,
        control_rostros,
    ],
    layout=widgets.Layout(width='100%', gap='6px'),
)
display(control_fidelidad)
display(control_escala)
display(panel_mejoras)


<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>🚀 04 · Restaurar la imagen</h2>
</div>

Cuando hayas elegido la foto y ajustado los parámetros, ejecuta la celda siguiente.

⏳ Espera a que aparezca el mensaje **«¡Restauración completada!»** antes de continuar.

In [ ]:
from datetime import datetime

ultimo_resultado = None
ultimo_original = None
ultimo_directorio = None

if not selector_imagen.value:
    raise ValueError(
        'Primero sube una imagen o coloca una imagen ' 
        'compatible en la carpeta '
        'entrada. Después vuelve a ejecutar la celda ' 
        'de subida (sección 2).'
    )

origen = Path(selector_imagen.value)
if not origen.is_file():
    raise FileNotFoundError(
        f'No existe la imagen seleccionada: {origen}'
    )

fidelidad = float(control_fidelidad.value)
escala = int(control_escala.value)
porcentaje = int(round(fidelidad * 100))
marca = (
    f"restauracion_{origen.stem}_w{porcentaje:02d}_{escala}x"
)
out_dir = RESULTADOS / marca
if out_dir.exists():
    # Evita sobrescribir resultados anteriores
    # de la misma configuración.
    marca += '_' + datetime.now().strftime('%Y%m%d_%H%M%S')
    out_dir = RESULTADOS / marca
out_dir.mkdir(parents=True, exist_ok=True)

comando = [
    sys.executable, str(SCRIPT),
    '-i', str(origen),
    '-o', str(out_dir),
    '-w', str(fidelidad),
    '-s', str(escala),
]
if control_fondo.value:
    comando.extend(['--bg_upsampler', 'realesrgan'])
if control_rostros.value:
    comando.append('--face_upsample')
print('Imagen:', origen.name)
print('Fidelidad:', fidelidad)
print('Ampliación:', f'{escala}×')
estado_fondo = (
    'activada' if control_fondo.value else 'desactivada'
)
print('Mejora del fondo:', estado_fondo)
estado_rostros = (
    'activada' if control_rostros.value else 'desactivada'
)
print('Mejora adicional de rostros:', estado_rostros)
print('\nProcesando…')
proceso = subprocess.run(
    comando,
    cwd=str(CODEFORMER),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(proceso.stdout)
if proceso.returncode != 0:
    raise RuntimeError(
        f'CodeFormer terminó con código {proceso.returncode}. '
        'Revisa el registro anterior.'
    )

carpeta_final = out_dir / 'final_results'
candidatos = (
    sorted(carpeta_final.glob('*'))
    if carpeta_final.is_dir() else []
)
candidatos = [
    p for p in candidatos
    if p.is_file() and p.suffix.lower() in EXTENSIONES
]
if not candidatos:
    raise FileNotFoundError(
        'El proceso terminó, pero no se encontró ' 
        'la imagen restaurada '
        'en final_results.'
    )
# El cuaderno procesa una imagen; toma su resultado.
coincidentes = [
    p for p in candidatos if p.stem == origen.stem
]
ultimo_resultado = (
    coincidentes[0] if coincidentes else candidatos[0]
)
ultimo_original = origen
ultimo_directorio = out_dir

if 'DRIVE_RESULTADOS' in globals():
    carpeta_drive = DRIVE_RESULTADOS / marca
    carpeta_drive.mkdir(parents=True, exist_ok=True)
    copia_drive = carpeta_drive / ultimo_resultado.name
    shutil.copy2(ultimo_resultado, copia_drive)
    print('\nCopia permanente en Drive:', copia_drive)

with Image.open(origen) as im:
    dimensiones_originales = im.size
with Image.open(ultimo_resultado) as im:
    dimensiones_restauradas = im.size
print('\n¡Restauración completada!')
print('Resultado:', ultimo_resultado)
print('Dimensiones originales:', dimensiones_originales)
print('Dimensiones restauradas:', dimensiones_restauradas)


<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>🖼️ 05 · Comparar el antes y el después</h2>
</div>

Revisa las dos imágenes una al lado de la otra para comprobar los cambios y la calidad del resultado.

In [ ]:
if (
    'ultimo_resultado' not in globals()
    or ultimo_resultado is None
    or not Path(ultimo_resultado).is_file()
):
    print(
        'Todavía no hay una restauración disponible. '
        'Ejecuta primero la celda de restauración.'
    )
else:
    with Image.open(ultimo_original) as im:
        original_rgb = im.convert('RGB').copy()
    with Image.open(ultimo_resultado) as im:
        restaurada_rgb = im.convert('RGB').copy()
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(original_rgb)
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(restaurada_rgb)
    axes[1].set_title('CodeFormer restaurada')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    nombre_resultado = Path(ultimo_resultado).name
    print(
        'Resultado listo para descargar:', nombre_resultado
    )


<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>📥 06 · Descargar la imagen restaurada</h2>
</div>

Ejecuta esta celda cuando estés conforme con el resultado. La imagen restaurada también se conserva en la carpeta de resultados de tu proyecto en Google Drive.

✅ Si todavía no hay una restauración disponible, vuelve a la sección **04 · Restaurar la imagen**.

In [ ]:
if (
    'ultimo_resultado' not in globals()
    or ultimo_resultado is None
    or not Path(ultimo_resultado).is_file()
):
    print(
        'No hay una imagen restaurada para descargar. '
        'Ejecuta primero la restauración y la comparación.'
    )
elif not EN_COLAB:
    print(
        'La imagen restaurada está guardada en:',
        ultimo_resultado,
    )
else:
    print('Descargando:', Path(ultimo_resultado).name)
    files.download(str(ultimo_resultado))
